# **OOP**

## **Creational Patterns Exercise**

Website with exercises https://courses.dallard.tech/sys_design/creational_pattern/ 

### Imports 

In [1]:
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Dict, Type, Any
import json

### **Exercise 1: Factory Pattern**

#### 1) The interface

Start by defining an Abstract Base Class (ABC). This ensures that every processor you build in the future "promises" to have the same methods. 

- **Method**: validate(details: dict) -> bool
- **Method**: process(amount: float, details: dict) -> dict

We start by creating a common blueprint for all payment processors.

In [2]:
class PaymentProcessor(ABC):    #creates common interface for all payment processors
    @abstractmethod
    def validate(self, details: dict) -> bool:   #for invalid details
        pass

    @abstractmethod
    def process(self, amount: float, details: dict) -> dict:
        pass

#### 2) Concrete Implementations

Create a specific class for CreditCardProcessor, BankTransferProcessor, and PayPalProcessor. Move the specific logic (like the 2.9% fee for cards) into these classes.

In [3]:
class CreditCardProcessor(PaymentProcessor):
    def validate(self, details: dict) -> bool:
        num = details.get("card_number", "")
        cvv = details.get("cvv", "")
        if not num or len(num) != 16:
            return False
        if not cvv or len(cvv) != 3:
            return False 
        return True
       
    def process(self, amount: float, details: dict) -> dict:
        if amount <= 0:
            return {"success": False, "error": "Amount must be positive"}
        if not self.validate(details):
            return {"success": False, "error": "Invalid credit card details"}

        fee = amount * 0.029
        return {"success": True, "method": "credit_card", "amount": amount + fee, "fee": fee}


class BankTransferProcessor(PaymentProcessor):
    def validate(self, details: dict) -> bool:
        iban = details.get("iban", "")
        if not iban or len(iban) < 15:
            return False
        return True

    def process(self, amount: float, details: dict) -> dict:
        if amount <= 0:
            return {"success": False, "error": "Amount must be positive"}
        if not self.validate(details):
            return {"success": False, "error": "Invalid IBAN"}

        fee = 1.50
        return {"success": True, "method": "bank_transfer", "amount": amount + fee, "fee": fee}


class PayPalProcessor(PaymentProcessor):
    def validate(self, details: dict) -> bool:
        email = details.get("email", "")
        if not email or "@" not in email:
            return False 
        return True

    def process(self, amount: float, details: dict) -> dict:
        if amount <= 0:
            return {"success": False, "error": "Amount must be positive"}
        if not self.validate(details):
            return {"success": False, "error": "Invalid PayPal email"}

        fee = amount * 0.034 + 0.30
        return {"success": True, "method": "paypal", "amount": amount + fee, "fee": fee}



#### 3) The Factory

Create the PaymentFactory. This class acts as the "Middleman." It takes a string (like "paypal") and returns an instance of the corresponding processor.

In [4]:
class PaymentFactory:

    def __init__(self):
        self._processors = {
            "credit_card": CreditCardProcessor,
            "bank_transfer": BankTransferProcessor,
            "paypal": PayPalProcessor,}

    def get_processor(self, payment_type: str):
        processor_class = self._processors.get(payment_type)
        if not processor_class:
            raise ValueError(f"Unknown payment type: {payment_type}")

        return processor_class()

TEST

In [5]:
factory = PaymentFactory()
processor = factory.get_processor("credit_card")
result = processor.process(100.0, {"card_number": "3141592653589793", "cvv": "567"})

print(result)

{'success': True, 'method': 'credit_card', 'amount': 102.9, 'fee': 2.9000000000000004}


### **Exercise 2: Builder Pattern**

#### The Product

Define an Employee class. This class should be a simple data container (or a Data Class) that holds the final state of the record.

In [6]:
@dataclass
class Employee:
    first_name: str
    last_name: str
    email: str
    department: str = ""
    position: str = ""
    salary: float = 0.0
    start_date: str = ""
    manager_id: Optional[int] = None
    phone: Optional[str] = None
    address: Optional[str] = None
    emergency_contact: Optional[str] = None
    has_parking: bool = False
    has_laptop: bool = False
    has_vpn_access: bool = False
    has_admin_rights: bool = False
    office_location: Optional[str] = None
    contract_type: str = "permanent"

#### The builder

Create an EmployeeBuilder. Instead of one massive function, create small, chainable methods.

Each method in a Builder usually returns self. This allows you to "chain" calls like .with_name().with_job().build().

In [7]:
class EmployeeBuilder:
    def __init__(self):
        self._data = {
            "has_parking": False,
            "has_laptop": False,
            "has_vpn_access": False,
            "has_admin_rights": False,
            "contract_type": "permanent",
        }

    def with_name(self, first_name: str, last_name: str) -> "EmployeeBuilder":
        self._data["first_name"] = first_name
        self._data["last_name"] = last_name
        return self

    def with_email(self, email: str) -> "EmployeeBuilder":
        if "@" not in email:
            raise ValueError("Valid email is required")
        self._data["email"] = email
        return self

    def with_job(self, department: str, position: str, salary: float,
                 start_date: str = "2024-01-01") -> "EmployeeBuilder":
        if salary < 0:
            raise ValueError("Salary cannot be negative")
        self._data["department"] = department
        self._data["position"] = position
        self._data["salary"] = salary
        self._data["start_date"] = start_date
        return self

    def with_contact(self, phone: str = None, address: str = None,
                     emergency_contact: str = None) -> "EmployeeBuilder":
        self._data["phone"] = phone
        self._data["address"] = address
        self._data["emergency_contact"] = emergency_contact
        return self

    def with_equipment(self, laptop: bool = False,
                       parking: bool = False) -> "EmployeeBuilder":
        self._data["has_laptop"] = laptop
        self._data["has_parking"] = parking
        return self

    def with_access(self, vpn: bool = False,
                    admin: bool = False) -> "EmployeeBuilder":
        self._data["has_vpn_access"] = vpn
        self._data["has_admin_rights"] = admin
        return self

    def with_manager(self, manager_id: int) -> "EmployeeBuilder":
        self._data["manager_id"] = manager_id
        return self

    def with_location(self, office_location: str,
                      contract_type: str = "permanent") -> "EmployeeBuilder":
        self._data["office_location"] = office_location
        self._data["contract_type"] = contract_type
        return self

    def build(self) -> Employee:
        required = ["first_name", "last_name", "email",
                    "department", "position", "salary", "start_date"]
        for field_name in required:
            if field_name not in self._data:
                raise ValueError(f"Missing required field: '{field_name}'")
        return Employee(**self._data)


# 3) PRESET BUILDER
class DeveloperBuilder(EmployeeBuilder):
    def __init__(self, first_name: str, last_name: str, email: str):
        super().__init__()
        self.with_name(first_name, last_name)
        self.with_email(email)
        self.with_equipment(laptop=True, parking=False)
        self.with_access(vpn=True, admin=False)
        self._data["department"] = "Engineering"
        self._data["contract_type"] = "permanent"




--> Test

In [8]:
if __name__ == "__main__":
    employee = (
        EmployeeBuilder()
        .with_name("Harry", "Kane")
        .with_email("Harry.Kane@football.com")
        .with_job("Engineering", "Senior Developer", 99000, "2022-01-10")
        .with_equipment(laptop=True, parking=False)
        .with_access(vpn=True, admin=True)
        .with_location("Manchester")
        .build()
    )

    intern = (
        EmployeeBuilder()
        .with_name("Caroline ", "Bessette")
        .with_email("caroline.bessette@contact.com")
        .with_job("Marketing", "Intern", 15000, "2025-02-08")
        .with_access(vpn=False, admin=False)
        .with_location("New York", contract_type="internship")
        .build()
    )

    emp2 = (
        DeveloperBuilder("Harry", "Kane", "Harry.kane@football.com")
        .with_job("Engineering", "Senior Developer", 99000)
        .build()
    )

    print(employee)
    print(intern)
    print(emp2)

Employee(first_name='Harry', last_name='Kane', email='Harry.Kane@football.com', department='Engineering', position='Senior Developer', salary=99000, start_date='2022-01-10', manager_id=None, phone=None, address=None, emergency_contact=None, has_parking=False, has_laptop=True, has_vpn_access=True, has_admin_rights=True, office_location='Manchester', contract_type='permanent')
Employee(first_name='Caroline ', last_name='Bessette', email='caroline.bessette@contact.com', department='Marketing', position='Intern', salary=15000, start_date='2025-02-08', manager_id=None, phone=None, address=None, emergency_contact=None, has_parking=False, has_laptop=False, has_vpn_access=False, has_admin_rights=False, office_location='New York', contract_type='internship')
Employee(first_name='Harry', last_name='Kane', email='Harry.kane@football.com', department='Engineering', position='Senior Developer', salary=99000, start_date='2024-01-01', manager_id=None, phone=None, address=None, emergency_contact=None,

### **Exercise 3:  Singleton Pattern**

**1) The singleton**

In [12]:
config_data = {
    "app": {"name": "PaymentPlatform", "debug": True},
    "database": {"host": "localhost", "port": 5432},
    "email": {"smtp_host": "smtp.company.com", "sender": "no-reply@company.com"},
    "payment": {"api_key": "sk_test_123456", "environment": "sandbox"}
}

with open("config.json", "w") as f:
    json.dump(config_data, f, indent=2)

print("config.json created ")

config.json created 


In [13]:
class ConfigManager:
    _instance = None
    _config = None
    _load_count = 0  

    def __init__(self):
        raise RuntimeError("Use ConfigManager.get_instance() instead")

    @classmethod
    def get_instance(cls) -> "ConfigManager":
        if cls._instance is None:
            cls._instance = cls.__new__(cls)
            cls._instance._load_config()
        return cls._instance

    def _load_config(self):
        with open("config.json", "r") as f:
            self._config = json.load(f)
        ConfigManager._load_count += 1
        print(f"Config loaded from file (load #{ConfigManager._load_count})")  

    def get(self, key: str, default=None):  
        keys = key.split(".")
        value = self._config
        for k in keys:
            if isinstance(value, dict) and k in value:
                value = value[k]
            else:
                return default 
        return value

    def reload(self):
        self._load_config()



**2) Refactored modules**

In [14]:
def connect_database():
    config = ConfigManager.get_instance()
    db_host = config.get("database.host")
    db_port = config.get("database.port")
    print(f"Connecting to database at {db_host}:{db_port}")

def send_email(to: str, subject: str):
    config = ConfigManager.get_instance()
    smtp_host = config.get("email.smtp_host")
    sender = config.get("email.sender")
    print(f"Sending email from {sender} via {smtp_host}")

def process_payment(amount: float):
    config = ConfigManager.get_instance()
    api_key = config.get("payment.api_key")
    environment = config.get("payment.environment")
    print(f"Processing {amount}€ in {environment} mode")

def start_application():
    config = ConfigManager.get_instance()
    app_name = config.get("app.name")
    debug = config.get("app.debug")
    print(f"Starting {app_name} (debug={debug})")

**3) Usage**

In [15]:
if __name__ == "__main__":
    start_application()
    connect_database()
    send_email("user@test.com", "Welcome")
    process_payment(99.99)

    c1 = ConfigManager.get_instance()
    c2 = ConfigManager.get_instance()
    print(f"\nSame instance? {c1 is c2}")
    missing = ConfigManager.get_instance().get("app.nonexistent", "N/A")
    print(f"Missing key fallback: {missing}")  

Config loaded from file (load #1)
Starting PaymentPlatform (debug=True)
Connecting to database at localhost:5432
Sending email from no-reply@company.com via smtp.company.com
Processing 99.99€ in sandbox mode

Same instance? True
Missing key fallback: N/A
